# Stage 2 — DeepAR v1

Trains SageMaker's built-in **DeepAR** algorithm on the 333 valid clients (>= 730 active days),
at hourly resolution, forecasting 1 week (168 hours) ahead.

Sequence: baseline job (CPU) -> small hyperparameter tuning job (CPU) -> single comparison job
(GPU) on the best config -> batch transform evaluation on the held-out week.

**Cost note**: every training/tuning job here is on the order of single-digit dollars-cents,
not dollars -- this is a small dataset (333 series, ~4 years hourly) and DeepAR is a lightweight
RNN. The GPU job exists to *compare* wall-clock/cost against CPU, not because this workload needs one.

In [ ]:
%pip install -q duckdb sagemaker boto3

In [ ]:
import json
from pathlib import Path

import boto3
import duckdb
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

REGION = boto3.Session().region_name or "us-east-1"
BUCKET = "<your-bucket>"
RAW_PREFIX = "ts-forecast-demo/raw/electricity"
CURATED_PREFIX = "ts-forecast-demo/curated"
DEEPAR_PREFIX = "ts-forecast-demo/deepar-v1"
SAGEMAKER_ROLE = "arn:aws:iam::<ACCOUNT_ID>:role/ts-forecast-demo-sagemaker-role"

FREQ = "H"
PREDICTION_LENGTH = 24 * 7   # 1 week ahead
CONTEXT_LENGTH = 24 * 7      # 1 week of lookback (baseline default; tuned later)

session = sagemaker.Session()
s3 = boto3.client("s3")

## 1. Resample to hourly via DuckDB, filtered to valid clients

Same pushdown pattern as the EDA notebook: the aggregation runs in DuckDB against S3, only the
already-hourly result (333 clients x ~4 years x 24h ~= 10M rows) comes into pandas -- an order of
magnitude smaller than the raw 15-min table that caused the stage-0 OOM.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
creds = boto3.Session().get_credentials().get_frozen_credentials()
con.execute(f"SET s3_region='{REGION}';")
con.execute(f"SET s3_access_key_id='{creds.access_key}';")
con.execute(f"SET s3_secret_access_key='{creds.secret_key}';")
if creds.token:
    con.execute(f"SET s3_session_token='{creds.token}';")

RAW_GLOB = f"s3://{BUCKET}/{RAW_PREFIX}/year=*/*.parquet"
METADATA_PATH = f"s3://{BUCKET}/{CURATED_PREFIX}/client_metadata.parquet"

hourly = con.sql(f"""
    SELECT
        r.client_id,
        time_bucket(INTERVAL '1 hour', r.timestamp) AS ts_hour,
        sum(r.kwh) AS kwh
    FROM read_parquet('{RAW_GLOB}', hive_partitioning=1) r
    JOIN read_parquet('{METADATA_PATH}') m USING (client_id)
    WHERE m.is_valid = true
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

activation = con.sql(f"""
    SELECT client_id, first_active_ts
    FROM read_parquet('{METADATA_PATH}')
    WHERE is_valid = true
""").df().set_index("client_id")["first_active_ts"]

print(hourly.shape, hourly["client_id"].nunique())

## 2. Build DeepAR JSON Lines

Each client's series is reindexed to a full hourly range from its own `first_active_ts` onward,
gaps filled with `NaN` (DeepAR's required missing-value token -- a literal `"NaN"` string in the
JSON, not `null`). `train.json` truncates the last `PREDICTION_LENGTH` hours; `test.json` keeps
the full series so we can score against the truncated tail later.

In [ ]:
def to_deepar_target(values: np.ndarray) -> list:
    """DeepAR wants NaN encoded as the literal string 'NaN', not JSON null."""
    return ["NaN" if pd.isna(v) else round(float(v), 4) for v in values]


client_ids = sorted(hourly["client_id"].unique())
client_index = {cid: i for i, cid in enumerate(client_ids)}

train_records, test_records = [], []

for cid, grp in hourly.groupby("client_id"):
    start = activation[cid].floor("h")
    full_range = pd.date_range(start, grp["ts_hour"].max(), freq="h")
    series = grp.set_index("ts_hour")["kwh"].reindex(full_range)

    record = {
        "start": start.strftime("%Y-%m-%d %H:%M:%S"),
        "target": to_deepar_target(series.values),
        "cat": [client_index[cid]],
    }
    test_records.append(record)

    train_record = dict(record)
    train_record["target"] = record["target"][:-PREDICTION_LENGTH]
    train_records.append(train_record)

print(f"{len(train_records)} series prepared")

In [ ]:
local_dir = Path("deepar_data")
local_dir.mkdir(exist_ok=True)

for name, records in [("train", train_records), ("test", test_records)]:
    path = local_dir / f"{name}.json"
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    s3.upload_file(str(path), BUCKET, f"{DEEPAR_PREFIX}/{name}/{name}.json")

train_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/train/"
test_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/test/"
print(train_s3, test_s3)

## 3. Baseline training job (CPU)

Sanity-check the data format and get a first working model before spending anything on tuning.

In [ ]:
deepar_image = retrieve("forecasting-deepar", REGION)

baseline_hyperparameters = {
    "time_freq": FREQ,
    "context_length": str(CONTEXT_LENGTH),
    "prediction_length": str(PREDICTION_LENGTH),
    "num_cells": "40",
    "num_layers": "2",
    "epochs": "30",
    "mini_batch_size": "64",
    "learning_rate": "1e-3",
    "likelihood": "gaussian",
    "early_stopping_patience": "10",
}

baseline_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-baseline",
    hyperparameters=baseline_hyperparameters,
    sagemaker_session=session,
)

baseline_estimator.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

## 4. Hyperparameter tuning (small budget, CPU)

6 jobs, 2 in parallel -- enough to see tuning move the needle, not a real production search.
Objective: `test:mean_wQuantileLoss` (DeepAR's built-in probabilistic accuracy metric),
lower is better.

In [ ]:
tuning_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-hpo",
    hyperparameters={
        "time_freq": FREQ,
        "prediction_length": str(PREDICTION_LENGTH),
        "early_stopping_patience": "10",
    },
    sagemaker_session=session,
)

hyperparameter_ranges = {
    "context_length": IntegerParameter(24 * 3, 24 * 14),
    "num_cells": IntegerParameter(30, 100),
    "num_layers": IntegerParameter(1, 3),
    "learning_rate": ContinuousParameter(1e-4, 1e-2),
}

tuner = HyperparameterTuner(
    estimator=tuning_estimator,
    objective_metric_name="test:mean_wQuantileLoss",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=6,
    max_parallel_jobs=2,
)

tuner.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

In [ ]:
best_job = tuner.best_training_job()
best_hyperparameters = sagemaker.estimator.Estimator.attach(best_job, sagemaker_session=session).hyperparameters()
print(best_job)
print(best_hyperparameters)

## 5. GPU comparison run

Same best hyperparameters, different instance type -- the point is comparing wall-clock and cost,
not searching for a better model. Check the training job's billable seconds afterward
(`describe_training_job` -> `TrainingTimeInSeconds`) against the CPU baseline/HPO jobs.

In [ ]:
gpu_hyperparameters = {k: v for k, v in best_hyperparameters.items() if not k.startswith("_")}
gpu_hyperparameters["time_freq"] = FREQ
gpu_hyperparameters["prediction_length"] = str(PREDICTION_LENGTH)

gpu_estimator = Estimator(
    image_uri=deepar_image,
    role=SAGEMAKER_ROLE,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/model-gpu",
    hyperparameters=gpu_hyperparameters,
    sagemaker_session=session,
)

gpu_estimator.fit(
    {"train": TrainingInput(train_s3, content_type="json"), "test": TrainingInput(test_s3, content_type="json")}
)

In [ ]:
sm = boto3.client("sagemaker")

for name, job_name in [
    ("baseline (CPU)", baseline_estimator.latest_training_job.name),
    ("best HPO (CPU)", best_job),
    ("comparison (GPU)", gpu_estimator.latest_training_job.name),
]:
    desc = sm.describe_training_job(TrainingJobName=job_name)
    print(f"{name:20s} instance={desc['ResourceConfig']['InstanceType']:15s} "
          f"billable_seconds={desc['TrainingTimeInSeconds']}")

## 6. Evaluate via batch transform (not a persistent endpoint)

Batch transform bills only for the job's runtime -- appropriate here since we're scoring the
held-out week once, not serving live traffic. The transform input is `train.json` (target already
truncated); DeepAR forecasts `PREDICTION_LENGTH` steps forward from each series' end.

In [ ]:
best_estimator = sagemaker.estimator.Estimator.attach(best_job, sagemaker_session=session)

transformer = best_estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET}/{DEEPAR_PREFIX}/batch-eval",
    strategy="SingleRecord",
    accept="application/jsonlines",
)

transformer.transform(
    data=train_s3,
    content_type="application/jsonlines",
    split_type="Line",
)
transformer.wait()

In [ ]:
import io

output_key = f"{DEEPAR_PREFIX}/batch-eval/train.json.out"
obj = s3.get_object(Bucket=BUCKET, Key=output_key)
predictions = [json.loads(line) for line in obj["Body"].read().decode().splitlines()]

errors = []
for record, pred in zip(test_records, predictions):
    actual_tail = np.array([np.nan if v == "NaN" else v for v in record["target"][-PREDICTION_LENGTH:]])
    mean_pred = np.array(pred["quantiles"]["0.5"])  # median forecast
    mask = ~np.isnan(actual_tail)
    if mask.sum() > 0:
        rmse = np.sqrt(np.mean((actual_tail[mask] - mean_pred[mask]) ** 2))
        errors.append(rmse)

print(f"Mean RMSE across {len(errors)} clients: {np.mean(errors):.2f}")
print(f"Median RMSE: {np.median(errors):.2f}")

## Cost recap

Check each job's billable seconds (cell above) x its instance's on-demand rate. No endpoints were
left running -- batch transform tears down its instance automatically when the job completes.
Confirm in the SageMaker console (Training jobs / Batch transform jobs) that nothing shows `InProgress`
after this notebook finishes.